# Витрина эквайринга на уровне договора (lake-only по логике DAG техлида)

Самодостаточная тетрадка, повторяющая логику DAG `acquiring__merchants_stats` техлида.

**Гранулярность**: 1 строка = 1 `agr_id` (договор эквайринга в Альфа = `abs_agr_id` = `id` в R2).

**Источники**:
- Базовый периметр: `ods_alpha.scd1_agreements` + `ods.scd1_z_r2_ip_merchants` + `ods.scd1_z_client/z_cl_corp/z_branch` + `ods_alpha.scd1_companies`
- Тариф/комиссия/ВСП: R2-цепочка через `m.id = abs_agr_id`
- Lake-метрики: `scd1_trx + trx_acq + trx_int + base24_fiids` по логике `raw_trx.sql` + `agg_trx.sql`
- Терминалы: `scd1_pos_terminals` со срезом на конец месяца
- Сегмент клиента: `ocrm.s_org_ext + cdiul.ext_id_org` (по ИНН)
- Амортизация: `shestopalov_terminal_amortization_oneoff`

**Формулы**:
- `commission_total = commission_from_ops + commission_monthly`
- `chod = commission_total - int_component`
- `aur = retl_cnt * 1926`
- `fin_result = chod - aur - amortization`

In [ ]:
import pandas as pd

attributes_dict_df = pd.DataFrame([
    {'attribute': 'report_month', 'description_ru': 'Отчетный месяц витрины (метка YYYY-MM)'},
    {'attribute': 'snapshot_dt', 'description_ru': 'Дата среза для терминалов (последний день месяца)'},
    {'attribute': 'agr_id', 'description_ru': 'ID договора эквайринга (abs_agr_id Альфа = id в R2)'},
    {'attribute': 'n_agr', 'description_ru': 'Внутренний номер договора в Альфа'},
    {'attribute': 'c_agr_number', 'description_ru': 'Номер договора в Альфа'},
    {'attribute': 'n_cmp_client', 'description_ru': 'Номер компании в Альфа (n_cmp_client)'},
    {'attribute': 'inn', 'description_ru': 'ИНН клиента'},
    {'attribute': 'cmp_name', 'description_ru': 'Наименование клиента'},
    {'attribute': 'ssp_ocrm', 'description_ru': 'Сегмент/блок ОКРМ (по INN, через s_org_ext)'},
    {'attribute': 'cdi_id', 'description_ru': 'Идентификатор клиента в CDI (по INN)'},
    {'attribute': 'ogrn', 'description_ru': 'ОГРН клиента (z_cl_corp по cft_id)'},
    {'attribute': 'branch_cd', 'description_ru': 'Код филиала договора (через z_client.c_filial)'},
    {'attribute': 'branch_nm', 'description_ru': 'Наименование филиала договора (через z_branch.c_shortlabel)'},
    {'attribute': 'mcc', 'description_ru': 'MCC код (max по мерчантам компании)'},
    {'attribute': 'vsp_name', 'description_ru': 'Наименование ВСП договора (через z_depart.c_name)'},
    {'attribute': 'vsp_code', 'description_ru': 'Код ВСП договора (через z_depart.c_code)'},
    {'attribute': 'tariff_name', 'description_ru': 'Тарифный план договора (через r2_tariff_plan.c_name)'},
    {'attribute': 'commission_monthly', 'description_ru': 'Фиксированная месячная комиссия (через r2_tariff_fix.c_summa)'},
    {'attribute': 'retl_cnt', 'description_ru': 'Количество торговых точек (count distinct retl_id)'},
    {'attribute': 'term_cnt', 'description_ru': 'Количество терминалов на snapshot_dt (count distinct c_nter)'},
    {'attribute': 'trx_cnt', 'description_ru': 'Количество операций за месяц'},
    {'attribute': 'trx_sum', 'description_ru': 'Сумма операций за месяц'},
    {'attribute': 'commission_from_ops', 'description_ru': 'Комиссия с операций (sum n_amt_tax)'},
    {'attribute': 'int_component', 'description_ru': 'IRF / interchange (sum n_amt_fee)'},
    {'attribute': 'commission_total', 'description_ru': 'Итоговая комиссия = commission_from_ops + commission_monthly'},
    {'attribute': 'aur', 'description_ru': 'АУР = retl_cnt * 1926'},
    {'attribute': 'amortization', 'description_ru': 'Амортизация терминалов (по shestopalov_terminal_amortization_oneoff)'},
    {'attribute': 'chod', 'description_ru': 'ЧОД = commission_total - int_component'},
    {'attribute': 'fin_result', 'description_ru': 'Финансовый результат = chod - aur - amortization'},
    {'attribute': 'd_valid_from', 'description_ru': 'Дата начала действия договора'},
    {'attribute': 'd_valid_to', 'description_ru': 'Дата окончания действия договора'},
])

attributes_dict_df

## 1) Загрузка библиотек

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

## 2) Подключение к Impala

Используем тот же шаблон Kerberos-подключения, что в `datamart_month_acquiring.ipynb`.

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 3) Параметры запуска

Меняем только `report_month` (первый день нужного месяца). Имя выходного CSV формируется автоматически.

Также заранее обновляем метаданные источников (`invalidate metadata`), чтобы снизить риск падений из-за устаревших мета.

In [ ]:
# Единственный параметр (первый день месяца): '2026-01-01' / '2026-02-01' / '2026-03-01' / '2026-05-01'
report_month = '2026-01-01'

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_dt = month_end

output_csv_path = f'./datamart_month_lake_agr_{report_month_ts.strftime("%Y_%m")}.csv'

# Источники для обновления метаданных
invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals',
    'ods_alpha.scd1_trx',
    'ods_alpha.scd1_trx_acq',
    'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids',
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_client',
    'ods.scd1_z_cl_corp',
    'ods.scd1_z_branch',
    'ods.scd1_z_depart',
    'ods.scd1_z_r2_tariff_plan',
    'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix',
    'ocrm_ul.s_org_ext',
    'cdiul.ext_id_org',
    'sandbox_ai.shestopalov_terminal_amortization_oneoff',
]

invalidate_ok = 0
invalidate_failed = []
with imp:
    for t in invalidate_tables:
        try:
            imp.execute(f'invalidate metadata {t}')
            invalidate_ok += 1
        except Exception as e:
            invalidate_failed.append((t, str(e)))

print(f'Отчетный месяц: {report_month_label}')
print(f'month_start={month_start}, month_end={month_end}, snapshot_dt={snapshot_dt}')
print(f'Выходной CSV: {output_csv_path}')
print(f'Invalidate успешно: {invalidate_ok}/{len(invalidate_tables)}')
if invalidate_failed:
    print(f'Invalidate с ошибкой: {len(invalidate_failed)} (продолжаем выполнение)')
    pd.DataFrame(invalidate_failed, columns=['table_name', 'error_message']).head(20)

## 4) Базовый периметр (per `agr_id`, без фан-аута)

SQL по архитектуре `load_merchants.sql` техлида, но без джойна `alpha_merch` (чтобы не размножать строки на N мерчантов компании).

Цепочка:
- `ods_alpha.scd1_agreements` (фильтр `acq_class = 'SA'`, `abs_agr_id is not null`),
- `ods.scd1_z_r2_ip_merchants` через `m.id = abs_agr_id` (1-к-1),
- `ods.scd1_z_client / z_cl_corp / z_branch` через общий `id = m.c_cl_org` (cft_id),
- `ods_alpha.scd1_companies` через `c.n_cmp = a.n_cmp_client`.

Итог: `agr_id`, `n_agr`, `c_agr_number`, `n_cmp_client`, `inn`, `cmp_name`, `ogrn`, `branch_cd`, `branch_nm`, `d_valid_from`, `d_valid_to`.

In [ ]:
sql_base = f"""
with alpha_agreem as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.c_agr_number as string) as c_agr_number,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as string) as d_valid_from,
    cast(a.d_valid_to as string) as d_valid_to
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
    and coalesce(a.ods_deleted_flg, '0') <> '1'
),
ods_r2_merch as (
  select
    cast(m.id as string) as agr_id,
    cast(m.c_cl_org as string) as cft_id,
    cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
    cast(corp.c_register_date_reg as string) as egr_reg_dt,
    cast(br.c_code as string) as branch_cd,
    cast(br.c_shortlabel as string) as branch_nm
  from ods.scd1_z_r2_ip_merchants m
  left join ods.scd1_z_client cl on cast(cl.id as string) = cast(m.c_cl_org as string)
  left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = cast(m.c_cl_org as string)
  left join ods.scd1_z_branch br on cast(br.id as string) = cast(cl.c_filial as string)
)
select distinct
  a.agr_id,
  a.n_agr,
  a.c_agr_number,
  a.n_cmp_client,
  a.d_valid_from,
  a.d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as cmp_name,
  r2m.ogrn,
  r2m.branch_cd,
  r2m.branch_nm,
  r2m.cft_id
from alpha_agreem a
left join ods_r2_merch r2m on r2m.agr_id = a.agr_id
left join ods_alpha.scd1_companies c
  on cast(c.n_cmp as string) = a.n_cmp_client
where coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    base_df = imp.fetch(sql_base)

if base_df is None:
    base_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr', 'c_agr_number', 'n_cmp_client',
        'd_valid_from', 'd_valid_to',
        'inn', 'cmp_name',
        'ogrn', 'branch_cd', 'branch_nm', 'cft_id'
    ])

if not base_df.empty:
    for c in ['agr_id', 'n_agr', 'c_agr_number', 'n_cmp_client', 'inn', 'cft_id']:
        base_df[c] = base_df[c].astype(str)
    base_df = base_df.drop_duplicates(subset=['agr_id'], keep='first')

print(f'Базовый периметр: {len(base_df):,} строк')
print(f'Уникальных agr_id: {base_df["agr_id"].nunique():,}')
print(f'Уникальных inn: {base_df["inn"].nunique():,}')
base_df.head(5)

## 5) MCC по `agr_id`

`mcc` живет в `ods_alpha.scd1_merchants` и привязан к компании (`n_cmp`), не к договору. У одной компании может быть несколько мерчантов с разным MCC. Берем `max(n_mcc)` как репрезентативное.

In [ ]:
n_cmp_ids = sorted(base_df.get('n_cmp_client', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
n_cmp_in = ', '.join([f"'{x}'" for x in n_cmp_ids]) if n_cmp_ids else "''"

sql_mcc = f"""
select
  cast(m.n_cmp as string) as n_cmp_client,
  cast(max(m.n_mcc) as string) as mcc
from ods_alpha.scd1_merchants m
where cast(m.n_cmp as string) in ({n_cmp_in})
  and coalesce(m.ods_deleted_flg, '0') <> '1'
group by cast(m.n_cmp as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    mcc_df = imp.fetch(sql_mcc)

if mcc_df is None:
    mcc_df = pd.DataFrame(columns=['n_cmp_client', 'mcc'])

if not mcc_df.empty:
    mcc_df['n_cmp_client'] = mcc_df['n_cmp_client'].astype(str)
    mcc_df['mcc'] = mcc_df['mcc'].astype(str)

print(f'MCC по компаниям: {len(mcc_df):,} строк')
mcc_df.head(5)

## 6) Тариф / комиссия / ВСП по `agr_id` (R2-цепочка по скрипту техлида)

SQL по скрипту техлида:
- `m.id = abs_agr_id` (= наш `agr_id`),
- `m.c_tariff_plan -> tp.id` (тарифный план),
- `m.c_tariff_plan -> tt.c_tariff_plan` (тарифные настройки),
- `tt.c_tariff -> tf.id` (фиксированная сумма),
- `m.c_depart -> dep.id` (ВСП).

Итог: `tariff_name`, `commission_monthly`, `vsp_code`, `vsp_name` per `agr_id`.

Используем `max(...)` для защиты от фан-аута, который возможен на уровне `tariff_tune`.

In [ ]:
agr_ids = sorted(base_df.get('agr_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in agr_ids]) if agr_ids else "''"

sql_tariff = f"""
select
  cast(m.id as string) as agr_id,
  cast(max(tp.c_name) as string) as tariff_name,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly,
  cast(max(dep.c_code) as string) as vsp_code,
  cast(max(dep.c_name) as string) as vsp_name
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_fix tf
  on cast(tf.id as string) = cast(tt.c_tariff as string)
left join ods.scd1_z_depart dep
  on cast(dep.id as string) = cast(m.c_depart as string)
where cast(m.id as string) in ({agr_in})
group by cast(m.id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    tariff_df = imp.fetch(sql_tariff)

if tariff_df is None:
    tariff_df = pd.DataFrame(columns=['agr_id', 'tariff_name', 'commission_monthly', 'vsp_code', 'vsp_name'])

if not tariff_df.empty:
    tariff_df['agr_id'] = tariff_df['agr_id'].astype(str)
    for c in ['tariff_name', 'vsp_code', 'vsp_name']:
        tariff_df[c] = tariff_df[c].astype(str)

print(f'Тариф/ВСП по agr_id: {len(tariff_df):,} строк')
tariff_df.head(5)

## 7) `retl_cnt` по `agr_id`

`retl_cnt = count(distinct retl_id)` через связь Альфа-мерчантов с договором по компании (`m.n_cmp = a.n_cmp_client`). Один договор имеет N мерчантов компании — count distinct по уникальным `c_nmrc`.

In [ ]:
sql_retl_cnt = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  count(distinct cast(m.c_nmrc as string)) as retl_cnt
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_merchants m
  on cast(m.n_cmp as string) = cast(a.n_cmp_client as string)
 and coalesce(m.ods_deleted_flg, '0') <> '1'
 and m.c_nmrc is not null
where cast(a.abs_agr_id as string) in ({agr_in})
group by cast(a.abs_agr_id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    retl_cnt_df = imp.fetch(sql_retl_cnt)

if retl_cnt_df is None:
    retl_cnt_df = pd.DataFrame(columns=['agr_id', 'retl_cnt'])

if not retl_cnt_df.empty:
    retl_cnt_df['agr_id'] = retl_cnt_df['agr_id'].astype(str)
    retl_cnt_df['retl_cnt'] = pd.to_numeric(retl_cnt_df['retl_cnt'], errors='coerce').astype('Int64')

print(f'retl_cnt по agr_id: {len(retl_cnt_df):,} строк')
retl_cnt_df.head(5)

## 8) `term_cnt` по `agr_id` со срезом на `snapshot_dt`

По логике `load_cnt_retl_terms.sql` техлида:
- терминал считается активным на `snapshot_dt` если `snapshot_dt between d_ter_install and coalesce(d_ter_close, '2999-12-31')`,
- `term_cnt = count(distinct c_nter)`.

In [ ]:
sql_term_cnt = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  count(distinct cast(t.c_nter as string)) as term_cnt
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_merchants m
  on cast(m.n_cmp as string) = cast(a.n_cmp_client as string)
 and coalesce(m.ods_deleted_flg, '0') <> '1'
 and m.c_nmrc is not null
left join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = cast(m.c_nmrc as string)
 and t.c_nter is not null
 and cast('{snapshot_dt}' as date)
       between cast(t.d_ter_install as date)
       and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date))
where cast(a.abs_agr_id as string) in ({agr_in})
group by cast(a.abs_agr_id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    term_cnt_df = imp.fetch(sql_term_cnt)

if term_cnt_df is None:
    term_cnt_df = pd.DataFrame(columns=['agr_id', 'term_cnt'])

if not term_cnt_df.empty:
    term_cnt_df['agr_id'] = term_cnt_df['agr_id'].astype(str)
    term_cnt_df['term_cnt'] = pd.to_numeric(term_cnt_df['term_cnt'], errors='coerce').astype('Int64')

print(f'term_cnt по agr_id: {len(term_cnt_df):,} строк')
term_cnt_df.head(5)

## 9) Транзакционные метрики по `agr_id` (`raw_trx.sql` + `agg_trx.sql` техлида)

Фильтры по `raw_trx.sql`:
- `c_trx_class = 'SA'`,
- `c_trx_type = 'S01'`,
- `c_nter is not null`,
- `ods_deleted_flg != '1'`,
- `cf_trx_stat != 'R'`,
- `trunc(d_trx_orig, 'MM') = month_start`,
- `c_fiid_grp = 'RSHB'` (через `scd1_base24_fiids`).

Метрики:
- `trx_cnt = count(distinct n_trx)`,
- `trx_sum = sum(n_amt_src)`,
- `commission_from_ops = sum(n_amt_tax)`,
- `int_component = sum(n_amt_fee)` из `scd1_trx_int`.

In [ ]:
n_agrs = sorted(base_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
n_agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

sql_trx = f"""
with trx_base as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.c_nter as string) as c_nter,
    cast(t.c_fiid_acq as string) as c_fiid_acq,
    cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  left join ods_alpha.scd1_base24_fiids fa
    on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
  where t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = cast('{month_start}' as date)
    and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  where cast(a.n_agr as string) in ({n_agr_in})
),
tj as (
  select
    ta.n_agr,
    ta.n_trx,
    tb.n_amt_src,
    ta.n_amt_tax
  from ta
  join trx_base tb on tb.n_trx = ta.n_trx
),
trx_agg as (
  select
    tj.n_agr,
    count(distinct tj.n_trx) as trx_cnt,
    sum(tj.n_amt_src) as trx_sum,
    sum(tj.n_amt_tax) as commission_from_ops,
    sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as int_component
  from tj
  left join ods_alpha.scd1_trx_int i on cast(i.n_trx as string) = tj.n_trx
  group by tj.n_agr
)
select
  cast(a.abs_agr_id as string) as agr_id,
  t.trx_cnt,
  t.trx_sum,
  t.commission_from_ops,
  t.int_component
from trx_agg t
join ods_alpha.scd1_agreements a
  on cast(a.n_agr as string) = t.n_agr
where cast(a.abs_agr_id as string) in ({agr_in})
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    trx_df = imp.fetch(sql_trx)

if trx_df is None:
    trx_df = pd.DataFrame(columns=['agr_id', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component'])

if not trx_df.empty:
    trx_df['agr_id'] = trx_df['agr_id'].astype(str)
    trx_df = trx_df.drop_duplicates(subset=['agr_id'], keep='first')

print(f'TRX-метрики по agr_id: {len(trx_df):,} строк')
trx_df.head(5)

## 10) Сегмент клиента (`ssp_ocrm`, `cdi_id`) по `inn`

Цепочка `INN -> ocrm.s_org_ext -> cdiul.ext_id_org` (как в основной тетрадке). Эти атрибуты живут на уровне юрлица — поэтому одинаковы для всех `agr_id` одного `inn`. Это корректно по структуре данных.

In [ ]:
inn_values = sorted([x for x in base_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_ocrm = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ocrm_df = imp.fetch(sql_ocrm)

if ocrm_df is None:
    ocrm_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'cdi_id'])

if not ocrm_df.empty:
    ocrm_df['inn'] = ocrm_df['inn'].astype(str)
    ocrm_df['cdi_id'] = ocrm_df['cdi_id'].astype(str)
    allowed_ssp_values = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    ocrm_df['ssp_ocrm'] = ocrm_df['ssp_ocrm'].where(ocrm_df['ssp_ocrm'].isin(allowed_ssp_values), None)
    ocrm_df = ocrm_df.drop_duplicates(subset=['inn'], keep='first')

print(f'OCRM/CDI по inn: {len(ocrm_df):,} строк')
ocrm_df.head(5)

## 11) Амортизация по компании (`n_cmp_client`)

Логика из основной тетрадки: `sum(amortization_monthly)` через `pos_terminals` по компании. На уровне `n_cmp_client` — дублируется при merge на `agr_id` (это корректно: амортизация терминалов компании одинакова для всех договоров той же компании).

In [ ]:
sql_amort = f"""
with m as (
  select
    cast(n_cmp as string) as n_cmp_client,
    cast(c_nmrc as string) as c_nmrc
  from ods_alpha.scd1_merchants
  where cast(n_cmp as string) in ({n_cmp_in})
    and c_nmrc is not null
    and coalesce(ods_deleted_flg, '0') <> '1'
)
select
  m.n_cmp_client,
  sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
from m
join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = m.c_nmrc
 and t.c_nter is not null
left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
  on cast(am.c_nter as string) = cast(t.c_nter as string)
group by m.n_cmp_client
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    amort_df = imp.fetch(sql_amort)

if amort_df is None:
    amort_df = pd.DataFrame(columns=['n_cmp_client', 'amortization'])

if not amort_df.empty:
    amort_df['n_cmp_client'] = amort_df['n_cmp_client'].astype(str)

print(f'Амортизация по n_cmp_client: {len(amort_df):,} строк')
amort_df.head(5)

## 12) Финальная сборка витрины

Объединяем все блоки в одну витрину `final_df` на уровне `agr_id` и применяем финансовые формулы:

- `commission_total = commission_from_ops + commission_monthly`
- `chod = commission_total - int_component`
- `aur = retl_cnt * 1926`
- `fin_result = chod - aur - amortization`

In [ ]:
final_df = base_df.copy()
print(f'Старт (base): {len(final_df):,}')

if not mcc_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(mcc_df, on='n_cmp_client', how='left')
    print(f'После merge MCC: {before:,} -> {len(final_df):,}')
else:
    final_df['mcc'] = None

if not tariff_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(tariff_df, on='agr_id', how='left')
    print(f'После merge Tariff/ВСП: {before:,} -> {len(final_df):,}')
else:
    for c in ['tariff_name', 'commission_monthly', 'vsp_code', 'vsp_name']:
        final_df[c] = None

if not retl_cnt_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(retl_cnt_df, on='agr_id', how='left')
    print(f'После merge retl_cnt: {before:,} -> {len(final_df):,}')
else:
    final_df['retl_cnt'] = None

if not term_cnt_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(term_cnt_df, on='agr_id', how='left')
    print(f'После merge term_cnt: {before:,} -> {len(final_df):,}')
else:
    final_df['term_cnt'] = None

if not trx_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(trx_df, on='agr_id', how='left')
    print(f'После merge TRX-метрик: {before:,} -> {len(final_df):,}')
else:
    for c in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        final_df[c] = None

if not ocrm_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(ocrm_df, on='inn', how='left')
    print(f'После merge OCRM/CDI: {before:,} -> {len(final_df):,}')
else:
    final_df['ssp_ocrm'] = None
    final_df['cdi_id'] = None

if not amort_df.empty and not final_df.empty:
    before = len(final_df)
    final_df = final_df.merge(amort_df, on='n_cmp_client', how='left')
    print(f'После merge amortization: {before:,} -> {len(final_df):,}')
else:
    final_df['amortization'] = None

# Финансовые формулы
cf_ops_num = pd.to_numeric(final_df.get('commission_from_ops'), errors='coerce').fillna(0)
cm_num     = pd.to_numeric(final_df.get('commission_monthly'),  errors='coerce').fillna(0)
ic_num     = pd.to_numeric(final_df.get('int_component'),       errors='coerce').fillna(0)
am_num     = pd.to_numeric(final_df.get('amortization'),        errors='coerce').fillna(0)
rc_num     = pd.to_numeric(final_df.get('retl_cnt'),            errors='coerce').fillna(0)

final_df['commission_total'] = cf_ops_num + cm_num
final_df['chod']             = final_df['commission_total'] - ic_num
final_df['aur']              = rc_num * 1926
final_df['fin_result']       = final_df['chod'] - final_df['aur'] - am_num

final_df['report_month'] = report_month_label
final_df['snapshot_dt']  = snapshot_dt

# Финальный порядок колонок
final_columns = [
    'report_month', 'snapshot_dt',
    'agr_id', 'n_agr', 'c_agr_number', 'n_cmp_client',
    'inn', 'cmp_name',
    'ssp_ocrm', 'cdi_id',
    'ogrn', 'branch_cd', 'branch_nm',
    'mcc',
    'vsp_name', 'vsp_code', 'tariff_name', 'commission_monthly',
    'retl_cnt', 'term_cnt',
    'trx_cnt', 'trx_sum',
    'commission_from_ops', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result',
    'd_valid_from', 'd_valid_to'
]

for col in final_columns:
    if col not in final_df.columns:
        final_df[col] = None

# Типизация: целые
for col in ['retl_cnt', 'term_cnt', 'trx_cnt']:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('Int64')

# Типизация: денежные/decimal
def _to_decimal(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
            'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    final_df[col] = final_df[col].map(_to_decimal)

final_df = final_df[final_columns].copy()

print(f'\nИтоговая витрина: {len(final_df):,} строк, {len(final_df.columns)} атрибутов')
final_df.head(5)

## 13) Контрольные суммы и заполняемость полей

Сверка с Excel-отчетом бизнеса делается по сумматорам ниже:
- `sum(trx_sum)`, `sum(commission_from_ops)`, `sum(int_component)` — оборот, комиссия, IRF за месяц,
- `sum(chod)`, `sum(fin_result)` — основные KPI.

In [ ]:
row_cnt = len(final_df)
agr_unique = final_df['agr_id'].nunique() if row_cnt else 0
inn_unique = final_df['inn'].dropna().nunique() if row_cnt else 0

print(f'Отчетный месяц: {report_month_label}')
print(f'Строк: {row_cnt:,}')
print(f'Уникальных agr_id: {agr_unique:,}')
print(f'Уникальных inn: {inn_unique:,}')

def _sum_decimal(series):
    s = pd.to_numeric(series, errors='coerce').fillna(0)
    return float(s.sum())

print('\nКонтрольные суммы за месяц:')
for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly',
            'int_component', 'commission_total', 'aur', 'amortization',
            'chod', 'fin_result', 'retl_cnt', 'term_cnt']:
    val = _sum_decimal(final_df[col])
    print(f'  sum({col}) = {val:,.2f}')

# Заполняемость
field_fill_stats = []
for col in final_df.columns:
    non_null = int(final_df[col].notna().sum())
    field_fill_stats.append({
        'field_name': col,
        'non_null_cnt': non_null,
        'null_cnt': int(row_cnt - non_null),
        'fill_rate_pct': round((non_null / row_cnt) * 100, 2) if row_cnt else 0.0
    })

fill_stats_df = pd.DataFrame(field_fill_stats).sort_values(
    ['fill_rate_pct', 'field_name'], ascending=[True, True]
).reset_index(drop=True)

fill_stats_df

## 14) Сохранение в CSV

Имя файла формируется автоматически: `datamart_month_lake_agr_{YYYY_MM}.csv`.

In [ ]:
output_path = Path(output_csv_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'CSV сохранен: {output_path.resolve()}')
print(f'Строк: {len(final_df):,}')
print(f'Атрибутов: {len(final_df.columns)}')